## SAR Imagery Import

In [ ]:
# Option 1: Box download
import torch
import pandas as pd
import numpy as np
import zipfile
import requests
from pathlib import Path
import os

os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['TORCH_USE_CUDA_DSA'] = '1'

BOX_URLS = {
    "os1": "https://virginia.box.com/shared/static/9tz0fm7kq6745qzgy22hia985uf6c3hz.zip",
    "os2": "https://virginia.box.com/shared/static/t5dt2l6dgc3oz8oeizj83e8938gx69yr.zip",
    "fs":  "https://virginia.box.com/shared/static/u1lmcljm3o75gseghv5acnd4417qrkn2.zip",
}

PROJECT_ROOT = Path("..").resolve()
DATA_DIR     = PROJECT_ROOT / "data" / "classification"
DATA_DIR.mkdir(parents=True, exist_ok=True)


def download_and_extract(key: str, out_dir: Path) -> None:
    url = BOX_URLS[key]
    if not url:
        print(f"[{key}] No URL provided, skipping.")
        return
    if out_dir.exists() and any(out_dir.iterdir()):
        print(f"[{key}] Already extracted, skipping.")
        return
    zip_path = DATA_DIR / f"{key}.zip"
    print(f"[{key}] Downloading...")
    dl_url = url + ("?dl=1" if "?" not in url else "&dl=1")
    with requests.get(dl_url, stream=True) as r:
        r.raise_for_status()
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
    print(f"[{key}] Extracting...")
    out_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf_data:
        zf_data.extractall(out_dir)
    zip_path.unlink()
    print(f"[{key}] Done -> {out_dir}")


download_and_extract("os1", DATA_DIR / "os1")
download_and_extract("os2", DATA_DIR / "os2")
download_and_extract("fs",  DATA_DIR / "fs")

opensar1 = pd.read_csv(DATA_DIR / "opensar1_labels.csv")
opensar2 = pd.read_csv(DATA_DIR / "opensar2_labels.csv")
fusar    = pd.read_csv(DATA_DIR / "fusar_labels.csv")

opensar1["path"] = opensar1["path"].apply(lambda p: str(PROJECT_ROOT / p))
opensar2["path"] = opensar2["path"].apply(lambda p: str(PROJECT_ROOT / p))
fusar["path"]    = fusar["path"].apply(lambda p: str(PROJECT_ROOT / p))

print(f"OpenSARShip 1: {len(opensar1)} rows")
print(f"OpenSARShip 2: {len(opensar2)} rows")
print(f"FuSARShip:     {len(fusar)} rows")

fusar.head()

In [ ]:
# Option 2: Google Drive download
import torch
import pandas as pd
import numpy as np
from pathlib import Path
import gdown
import os

os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['TORCH_USE_CUDA_DSA'] = '1'

DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/19jLMSzHChVLk-vVAg2muNN2OALzksWob"

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "classification"

if not (DATA_DIR / "opensar1_labels.csv").exists():
    print("Downloading classification folder from Google Drive...")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    gdown.download_folder(DRIVE_FOLDER_URL, output=str(DATA_DIR), quiet=False, use_cookies=False)
    print("Download complete.")
else:
    print(f"Data already present at {DATA_DIR}, skipping download.")

opensar1 = pd.read_csv(DATA_DIR / "opensar1_labels.csv")
opensar2 = pd.read_csv(DATA_DIR / "opensar2_labels.csv")
fusar    = pd.read_csv(DATA_DIR / "fusar_labels.csv")

opensar1["path"] = opensar1["path"].apply(lambda p: str(PROJECT_ROOT / p))
opensar2["path"] = opensar2["path"].apply(lambda p: str(PROJECT_ROOT / p))
fusar["path"]    = fusar["path"].apply(lambda p: str(PROJECT_ROOT / p))

print(f"OpenSARShip 1: {len(opensar1)} rows")
print(f"OpenSARShip 2: {len(opensar2)} rows")
print(f"FuSARShip:     {len(fusar)} rows")

fusar.head()

## Label Mapping and Filtering

In [ ]:
unique_labels = sorted(set(opensar1['label']).union(
                       set(opensar2['label'])).union(
                       set(fusar['label'])))

label_to_int = {label: i for i, label in enumerate(unique_labels)}

opensar1.loc[:, 'label_id'] = opensar1['label'].map(label_to_int)
opensar2.loc[:, 'label_id'] = opensar2['label'].map(label_to_int)
fusar.loc[:, 'label_id']    = fusar['label'].map(label_to_int)

print("Label mapping:", label_to_int)

opensar1 = opensar1[opensar1['label'] != 'unknown']
opensar2 = opensar2[opensar2['label'] != 'unknown']
fusar    = fusar[fusar['label'] != 'unknown']

print(f"OpenSARShip 1: {len(opensar1)} rows")
print(f"OpenSARShip 2: {len(opensar2)} rows")
print(f"FuSARShip:     {len(fusar)} rows")

class_names = [k for k, v in sorted(label_to_int.items(), key=lambda x: x[1]) if k != 'unknown']
print("class_names:", class_names)

## Dataset Creation

In [ ]:
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms
from PIL import Image


class create_dataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        try:
            image = Image.open(row['path']).convert('L')
        except (FileNotFoundError, OSError) as e:
            raise RuntimeError(f"Error loading image at index {idx}: {row['path']}") from e
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(row['label_id'], dtype=torch.long)


RESNET_MEAN = [0.485, 0.456, 0.406]
RESNET_STD  = [0.229, 0.224, 0.225]

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15, fill=0),
    transforms.ToTensor(),
    transforms.Normalize(mean=RESNET_MEAN, std=RESNET_STD)
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=RESNET_MEAN, std=RESNET_STD)
])

datasets_train = {
    'os1'  : create_dataset(opensar1, transform=transform_train),
    'os2'  : create_dataset(opensar2, transform=transform_train),
    'fusar': create_dataset(fusar,    transform=transform_train),
}
datasets_val = {
    'os1'  : create_dataset(opensar1, transform=transform_val),
    'os2'  : create_dataset(opensar2, transform=transform_val),
    'fusar': create_dataset(fusar,    transform=transform_val),
}

datasets_train['all'] = ConcatDataset(list(datasets_train.values()))
datasets_val['all']   = ConcatDataset(list(datasets_val.values()))

print(f"OpenSARShip 1: {len(datasets_train['os1'])} samples")
print(f"OpenSARShip 2: {len(datasets_train['os2'])} samples")
print(f"FuSARShip:     {len(datasets_train['fusar'])} samples")
print(f"Combined:      {len(datasets_train['all'])} samples")

## Train / Val / Test Split

In [ ]:
import zipfile as zf_mod
from torch.utils.data import Subset
from sklearn.model_selection import train_test_split

opensar1['source'] = 'os1'
opensar2['source'] = 'os2'
fusar['source']    = 'fusar'

domains = {
    'os1'  : {'df': opensar1},
    'os2'  : {'df': opensar2},
    'fusar': {'df': fusar},
    'all'  : {'df': pd.concat([opensar1, opensar2, fusar], ignore_index=True)},
}

for key in ['os1', 'os2', 'fusar']:
    df     = domains[key]['df']
    labels = df['label_id'].values

    trainval_idx, test_idx = train_test_split(
        range(len(df)), test_size=0.2, stratify=labels, random_state=42
    )
    train_idx, val_idx = train_test_split(
        trainval_idx, test_size=0.1, stratify=labels[trainval_idx], random_state=42
    )

    domains[key]['train_idx'] = list(train_idx)
    domains[key]['val_idx']   = list(val_idx)
    domains[key]['test_idx']  = list(test_idx)
    domains[key]['train_dataset'] = Subset(datasets_train[key], train_idx)
    domains[key]['val_dataset']   = Subset(datasets_val[key],   val_idx)
    domains[key]['test_dataset']  = Subset(datasets_val[key],   test_idx)

    print(f"Domain: {key}  train={len(train_idx)}  val={len(val_idx)}  test={len(test_idx)}")

os1_len = len(domains['os1']['df'])
os2_len = len(domains['os2']['df'])

all_train_idx = (
    list(domains['os1']['train_idx']) +
    [i + os1_len for i in domains['os2']['train_idx']] +
    [i + os1_len + os2_len for i in domains['fusar']['train_idx']]
)
all_val_idx = (
    list(domains['os1']['val_idx']) +
    [i + os1_len for i in domains['os2']['val_idx']] +
    [i + os1_len + os2_len for i in domains['fusar']['val_idx']]
)
all_test_idx = (
    list(domains['os1']['test_idx']) +
    [i + os1_len for i in domains['os2']['test_idx']] +
    [i + os1_len + os2_len for i in domains['fusar']['test_idx']]
)

full_train_ds = ConcatDataset(list(datasets_train.values())[:3])
full_val_ds   = ConcatDataset(list(datasets_val.values())[:3])

domains['all']['train_idx']     = all_train_idx
domains['all']['val_idx']       = all_val_idx
domains['all']['test_idx']      = all_test_idx
domains['all']['train_dataset'] = Subset(full_train_ds, all_train_idx)
domains['all']['val_dataset']   = Subset(full_val_ds,   all_val_idx)
domains['all']['test_dataset']  = Subset(full_val_ds,   all_test_idx)

print(f"Domain: all  train={len(all_train_idx)}  val={len(all_val_idx)}  test={len(all_test_idx)}")

# Save one npz per domain, then zip and remove
for key in domains:
    np.savez(
        f"misc/{key}_baseline_indices.npz",
        train_idx=np.array(domains[key]['train_idx']),
        val_idx=np.array(domains[key]['val_idx']),
        test_idx=np.array(domains[key]['test_idx']),
    )

indices_zip = Path("misc/baseline_indices.zip")
with zf_mod.ZipFile(indices_zip, "w", zf_mod.ZIP_DEFLATED) as zf_out:
    for key in domains:
        npz = Path(f"misc/{key}_baseline_indices.npz")
        if npz.exists():
            zf_out.write(npz, npz.name)
            npz.unlink()

print(f"Indices zipped to {indices_zip}")


In [ ]:
batch_size = 32

for key in domains:
    nw = 0 if key == 'fusar' else 2
    pw = key != 'fusar'
    pm = key != 'fusar'

    domains[key]['train_loader'] = DataLoader(
        domains[key]['train_dataset'], batch_size=batch_size, shuffle=True,
        num_workers=nw, pin_memory=pm, persistent_workers=pw
    )
    domains[key]['val_loader'] = DataLoader(
        domains[key]['val_dataset'], batch_size=batch_size, shuffle=False,
        num_workers=nw, pin_memory=pm, persistent_workers=pw
    )
    domains[key]['test_loader'] = DataLoader(
        domains[key]['test_dataset'], batch_size=batch_size, shuffle=False,
        num_workers=nw, pin_memory=pm, persistent_workers=pw
    )

    print(f"{key} — train: {len(domains[key]['train_loader'])} batches | "
          f"val: {len(domains[key]['val_loader'])} batches | "
          f"test: {len(domains[key]['test_loader'])} batches")

## Model Imports

In [ ]:
import sys
sys.path.insert(0, str(Path('.').resolve()))

from models.resnet_attention import resnet_attention
from models.resnet_transfer import CNN_resnet_transfer
from models.swin_transfer import swin_transfer

print("Models loaded:", resnet_attention, CNN_resnet_transfer, swin_transfer)

## Training

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
import time


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, reduction='mean'):
        super().__init__()
        self.gamma     = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss    = F.cross_entropy(inputs, targets, reduction='none')
        pt         = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


def train_model(model, save_path, class_names, train_loader, val_loader, train_labels,
                epochs=15, lr=1e-4, use_adamw=False, warmup=False):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model  = model.to(device)
    print(device)
    if torch.cuda.is_available():
        print(torch.cuda.get_device_name(0))
    print(next(model.parameters()).device)

    criterion = FocalLoss(gamma=2.0)

    optimizer_cls = AdamW if use_adamw else Adam
    weight_decay  = 1e-2  if use_adamw else 1e-4
    optimizer = optimizer_cls(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=weight_decay
    )

    if warmup:
        warmup_sched = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=2)
        cosine_sched = CosineAnnealingLR(optimizer, T_max=epochs - 2, eta_min=1e-6)
        scheduler    = SequentialLR(optimizer, schedulers=[warmup_sched, cosine_sched], milestones=[2])
    else:
        scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    best_val_acc = 0
    train_losses, val_losses = [], []
    train_accs,   val_accs   = [], []

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0, 0, 0

        for i, (inputs, labels) in enumerate(train_loader):
            t0 = time.time()
            inputs, labels = inputs.to(device), labels.to(device)
            t1 = time.time()
            optimizer.zero_grad(set_to_none=True)  # free gradient tensors rather than zero-filling
            outputs = model(inputs)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            t2 = time.time()

            if i == 0 and epoch == 0:
                print(f"Data->GPU: {t1-t0:.3f}s | Forward+backward: {t2-t1:.3f}s")

            running_loss += loss.item()
            _, predicted  = outputs.max(1)
            total        += labels.size(0)
            correct      += predicted.eq(labels).sum().item()

        scheduler.step()

        train_loss = running_loss / len(train_loader)
        train_acc  = 100. * correct / total

        # Free last batch's tensors before allocating the val pass
        del inputs, labels, outputs, loss, predicted

        model.eval()
        val_loss, correct, total = 0, 0, 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs        = model(inputs)
                loss           = criterion(outputs, labels)
                val_loss      += loss.item()
                _, predicted   = outputs.max(1)
                total         += labels.size(0)
                correct       += predicted.eq(labels).sum().item()

        val_loss   = val_loss / len(val_loader)
        val_acc    = 100. * correct / total
        current_lr = scheduler.get_last_lr()[0]

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)

        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"Val Acc: {val_acc:.2f}% | LR: {current_lr:.2e}")

    return model, {
        "train_loss": train_losses,
        "val_loss":   val_losses,
        "train_acc":  train_accs,
        "val_acc":    val_accs,
    }


In [ ]:
def sanity_check(model, label, train_loader, num_batches=5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model  = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

    model.train()
    for i, (inputs, labels) in enumerate(train_loader):
        if i >= num_batches:
            break
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        print(f"[{label}] Batch {i+1}/{num_batches} | Loss: {loss.item():.4f}")

    # Move back to CPU so the caller's del fully releases GPU memory
    model.cpu()


In [ ]:
# Model configurations — controls constructor kwargs and training hyperparams per model
MODEL_CONFIGS = {
    'bam': {
        'cls':       resnet_attention,
        'kwargs':    {'num_classes': len(class_names)},
        'epochs':    15,
        'lr':        1e-4,
        'use_adamw': False,
        'warmup':    False,
    },
    'res': {
        'cls':       CNN_resnet_transfer,
        'kwargs':    {'num_classes': len(class_names)},
        'epochs':    15,
        'lr':        1e-4,
        'use_adamw': False,
        'warmup':    False,
    },
    'vis': {
        'cls':       swin_transfer,
        'kwargs':    {'num_classes': len(class_names)},
        'epochs':    10,
        'lr':        5e-5,
        'use_adamw': True,
        'warmup':    True,
    },
}

In [ ]:
import gc

# Sanity check — verify each model forward-passes without error before full training
for tag, cfg in MODEL_CONFIGS.items():
    for domain in domains:
        model = cfg['cls'](**cfg['kwargs'])
        sanity_check(model, f"{domain}_{tag}", domains[domain]['train_loader'], num_batches=3)
        del model
        gc.collect()
        torch.cuda.empty_cache()


In [ ]:
import gc
import zipfile as zf_mod

all_histories = {}

for tag, cfg in MODEL_CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"  Training model: {tag.upper()}")
    print(f"{'='*60}")

    for domain in domains:
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        save_path    = f"weights/{domain}_{tag}_baseline.pth"
        train_labels = domains[domain]['df']['label_id'].values[domains[domain]['train_idx']]

        print(f"\n--- {tag} / {domain} ---")
        model = cfg['cls'](**cfg['kwargs'])

        trained_model, history = train_model(
            model, save_path, class_names,
            domains[domain]['train_loader'],
            domains[domain]['val_loader'],
            train_labels=train_labels,
            epochs=cfg['epochs'],
            lr=cfg['lr'],
            use_adamw=cfg['use_adamw'],
            warmup=cfg['warmup'],
        )

        all_histories[f"{tag}__{domain}"] = history
        trained_model.cpu()
        del trained_model, model
        gc.collect()
        torch.cuda.empty_cache()

# Zip all saved weights, then remove the individual .pth files
weights_zip = Path("weights/baseline_weights.zip")
with zf_mod.ZipFile(weights_zip, "w", zf_mod.ZIP_DEFLATED) as zf_out:
    for tag in MODEL_CONFIGS:
        for domain in domains:
            pth = Path(f"weights/{domain}_{tag}_baseline.pth")
            if pth.exists():
                zf_out.write(pth, pth.name)
                pth.unlink()

print(f"\nAll weights zipped to {weights_zip}")

In [ ]:
import matplotlib.pyplot as plt
import json

COLORS = {'os1': '#378ADD', 'os2': '#D4537E', 'fusar': '#1D9E75', 'all': '#BA7517'}

png_paths = []

for tag in MODEL_CONFIGS:
    keys = [k for k in all_histories if k.startswith(f"{tag}__")]
    fig, axes = plt.subplots(len(keys), 2, figsize=(12, 4 * len(keys)))
    fig.suptitle(f"{tag.upper()} Baseline Training History", fontsize=14, y=1.01)

    for i, key in enumerate(keys):
        domain = key.split("__")[1]
        h      = all_histories[key]
        color  = COLORS.get(domain, 'steelblue')
        ep     = range(1, len(h['train_loss']) + 1)

        ax1 = axes[i, 0]
        ax1.plot(ep, h['train_loss'], linestyle='dashed', color=color, label='train')
        ax1.plot(ep, h['val_loss'],   linestyle='solid',  color=color, label='val', linewidth=2)
        ax1.set_title(f"{domain} — loss")
        ax1.set_xlabel("Epoch")
        ax1.set_ylabel("Loss")
        ax1.legend()

        ax2 = axes[i, 1]
        ax2.plot(ep, h['train_acc'], linestyle='dashed', color=color, label='train')
        ax2.plot(ep, h['val_acc'],   linestyle='solid',  color=color, label='val', linewidth=2)
        ax2.set_title(f"{domain} — accuracy")
        ax2.set_xlabel("Epoch")
        ax2.set_ylabel("Accuracy (%)")
        ax2.legend()

    plt.tight_layout()
    png = Path(f"misc/{tag}_baseline_history.png")
    plt.savefig(png, dpi=150, bbox_inches='tight')
    plt.show()
    png_paths.append(png)

# Save history JSON then zip everything together
json_path = Path("misc/baseline_history.json")
with open(json_path, "w") as f:
    json.dump(all_histories, f, indent=2)

history_zip = Path("misc/baseline_history.zip")
with zf_mod.ZipFile(history_zip, "w", zf_mod.ZIP_DEFLATED) as zf_out:
    zf_out.write(json_path, json_path.name)
    json_path.unlink()
    for png in png_paths:
        if png.exists():
            zf_out.write(png, png.name)
            png.unlink()

print(f"History zipped to {history_zip}")
print("Keys:", list(all_histories.keys()))